# 11 · Parallelism inside NumPy and PyTorch

**Time:** 35–45 minutes. **Before this:** the GIL, pools, and measurement lessons.
**Requirements:** Python 3.11+, `requirements-advanced.txt`. PyTorch is optional.

You will distinguish Python workers from library threads and measure their
interaction. Arrays are small, every experiment is finite, and no GPU is needed.

[Course guide](README.md) · [Previous: Efficient multiprocessing](10_efficient_multiprocessing.ipynb) ·
[Next: Combining mechanisms](12_combining_concurrency_mechanisms.ipynb)

## 1. A Python call can contain work written in another language

**Native code** is compiled code, commonly written in C or C++. NumPy and
PyTorch use it for many numerical operations. **Vectorization** means applying
an operation to a whole array through a library call instead of writing a
Python loop for every value.

Many NumPy numerical operations release the GIL while doing native work.
Some use internal threads; others do not. Releasing the GIL and starting
several library threads are different behaviors. Object arrays and some
operations keep the GIL. An **object array** stores references to Python objects
instead of plain numerical values. Do not generalize one benchmark to every library call.
[NumPy thread behavior](https://numpy.org/doc/stable/reference/thread_safety.html).

In [1]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import json
import statistics
import subprocess
import sys
from time import perf_counter, process_time

import numpy as np
from queue import Queue
from threadpoolctl import ThreadpoolController, threadpool_info, threadpool_limits

print("NumPy:", np.__version__)
print("Detected native pools:")
for pool in threadpool_info():
    print({key: pool.get(key) for key in ("user_api", "internal_api", "num_threads")})

NumPy: 2.2.6
Detected native pools:
{'user_api': 'blas', 'internal_api': 'openblas', 'num_threads': 16}


`threadpoolctl` reports supported native thread pools. **BLAS** is a family of
numerical routines used for operations such as matrix multiplication. Your
installed NumPy build determines which BLAS implementation it uses.

If this list is empty, NumPy still works, but this tool may have no supported
pool to control in your build. Do not claim a thread setting took effect
without checking it. The examples will report the detected settings.
[threadpoolctl documentation](https://github.com/joblib/threadpoolctl).

## 2. Vectorization before adding workers

We will square integers and sum their remainders. The Python and NumPy
versions do the same arithmetic. NumPy's `int64` stores each integer in 64 bits
and has a fixed range. **Integer overflow** means a calculation goes outside that
range; unlike Python's ordinary integers, these values cannot grow indefinitely.
Our small inputs and their squared values stay within range.

**Predict:** can a whole-array operation be fast even with only one native thread?

In [2]:
values = np.arange(100_000, dtype=np.int64)
python_values = values.tolist()  # Convert once, outside the comparison.

def python_score():
    total = 0
    for value in python_values:
        total += (value * value) % 97
    return total

def numpy_score():
    return int(((values * values) % 97).sum())

def time_score(function):
    seconds, answers = [], []
    for _ in range(3):
        start = perf_counter()
        answers.append(function())
        seconds.append(perf_counter() - start)
    assert len(set(answers)) == 1
    return statistics.median(seconds), answers[0]

with threadpool_limits(limits=1, user_api="blas"):
    python_seconds, python_answer = time_score(python_score)
    numpy_seconds, numpy_answer = time_score(numpy_score)
assert python_answer == numpy_answer
print(f"Python loop: {python_seconds:.6f} s")
print(f"NumPy array operations: {numpy_seconds:.6f} s")
print("Matching answer:", python_answer)

Python loop: 0.002942 s
NumPy array operations: 0.000404 s
Matching answer: 4800196


Vectorization can reduce Python instruction overhead without using several
cores. **Elementwise** means applying the same arithmetic separately to each array
value. This example is not a BLAS operation; limiting BLAS threads
is not what controls its elementwise work. The next experiment specifically
uses matrix multiplication to study BLAS threads.

The array expression also creates temporary arrays. Faster execution can
have a memory cost. Include input conversion in a real timing if the real
application needs that conversion every time.

## 3. Two places to add threads

A matrix is a two-dimensional array. `left @ right` multiplies two matrices.
A BLAS library may divide that work among its own threads.

```text
Your Python pool:     worker 0                  worker 1
                          ↓                        ↓
Native matrix work:       BLAS library thread pool(s)
```

**Oversubscription** means more runnable threads compete than the available
CPU capacity can use effectively. A **backend** is the library implementation
doing the numerical work, such as OpenBLAS or MKL. Some backends give each calling thread its
own native pool; others share a pool across the process. With separate pools,
four Python workers each requesting four native threads could request roughly
16 compute threads, plus coordination. A shared pool can behave differently.

**Scheduling** is the operating system choosing which threads run. Extra
competition can increase its costs and strain **memory bandwidth**, the rate at
which data moves between RAM and the CPU. Inspect your installed backend rather than assuming one thread model.

In [3]:
rng = np.random.default_rng(7)
left = rng.normal(size=(320, 320))
right = rng.normal(size=(320, 320))
# Workers only read the input arrays. Each multiplication returns a new array.
left.flags.writeable = False
right.flags.writeable = False

def matrix_job(job_id):
    result = left @ right
    return job_id, float(result[0, 0])

with threadpool_limits(limits=1, user_api="blas"):
    expected_value = float((left @ right)[0, 0])

Shared arrays are safe here because workers only read them. Each worker owns
its result array. Mutating a shared NumPy array from several threads needs
coordination, even when native code releases the GIL.

Native limits can affect the whole process or only the calling thread. We use
an **initializer**, a function called once when each Python worker starts,
to apply the same limit in every worker. An outer context restores settings
after the pool finishes. Workers also report the settings they see. These are
configured limits, not measurements of how many threads actually did useful work.

A **warm-up** is an untimed first call that lets the library do initial setup.
Below, it happens in the main thread before timing each configuration.
We return one matrix value per job to keep results small. `np.allclose` checks
those values with a small allowance for rounding; this checks the experiment
without claiming to validate every element of the matrix.

`ThreadpoolController` inspects already-loaded supported libraries. We create
it after importing NumPy. Avoid competing limits or changing configuration in
every job. [Threadpoolctl's guidance for Python threads](https://github.com/joblib/threadpoolctl#usage-for-python-threads-restricting-controlled-library-thread-pool-sizes).

In [4]:
def measure_matrix_configuration(python_workers, native_threads, jobs=32):
    runs, cpu_runs = [], []
    worker_settings = Queue()
    controller = ThreadpoolController()

    def initialize_worker():
        controller.limit(limits=native_threads, user_api="blas")
        limits = tuple(p["num_threads"] for p in controller.info()
                       if p["user_api"] == "blas")
        worker_settings.put(limits)

    with controller.limit(limits=native_threads, user_api="blas"):
        matrix_job(-1)  # Untimed main-thread warm-up.
        for _ in range(3):
            wall_start, cpu_start = perf_counter(), process_time()
            with ThreadPoolExecutor(max_workers=python_workers,
                                    initializer=initialize_worker) as pool:
                results = list(pool.map(matrix_job, range(jobs)))
            runs.append(perf_counter() - wall_start)
            cpu_runs.append(process_time() - cpu_start)
            assert [item[0] for item in results] == list(range(jobs))
            assert np.allclose([item[1] for item in results], expected_value)
    # Worker threads have joined; nothing can add more observations now.
    observations = []
    while not worker_settings.empty():
        observations.append(worker_settings.get())
    observed = [list(limits) for limits in sorted(set(observations))]
    return statistics.median(runs), statistics.median(cpu_runs), observed

**Predict:** must two Python workers with two native threads each beat two
workers with one native thread each? Write your prediction, then compare.

The benchmark includes Python pool creation and shutdown, but excludes native
warm-up and array generation. Initializers are included. A main-thread warm-up
does not remove every per-worker initialization cost. Four small configurations
keep the experiment manageable. A real tuning decision needs representative matrix sizes and hardware.

In [5]:
print("Python workers | requested BLAS threads | worker observations | elapsed ms | CPU ms")
matrix_measurements = []
for workers, native in ((1, 1), (1, 2), (2, 1), (2, 2)):
    seconds, cpu_seconds, observed = measure_matrix_configuration(workers, native)
    matrix_measurements.append((workers, native, seconds))
    print(f"{workers:>14} | {native:>22} | {str(observed):>8} | "
          f"{seconds * 1000:>10.2f} | {cpu_seconds * 1000:>6.2f}")

Python workers | requested BLAS threads | worker observations | elapsed ms | CPU ms
             1 |                      1 |    [[1]] |      35.71 |  31.25


             1 |                      2 |    [[2]] |      22.99 |  31.25


             2 |                      1 |    [[1]] |      18.23 |  31.25
             2 |                      2 |    [[2]] |      12.68 |  62.50


There is no universal winning row. Small matrices can spend more time on
thread coordination than useful parallel work. Large operations may benefit
from internal threads. Separate independent jobs may benefit from outer workers.

CPU time counts all threads in this process. If it exceeds elapsed time, that
is consistent with multiple cores doing work. The ratio is not an exact
physical-core count. Short measurements and background activity add noise.

This sweep demonstrates competing layers of parallelism; it does not guarantee
oversubscription on your machine. With enough available cores, four compute
threads might fit comfortably. The practice cell tries `(2, 4)` as one extra
small configuration. For serious tuning, use more trials and vary the order of
configurations so warm caches or background activity do not consistently favor one row.

CPU-clock readings can be coarse, especially for very short intervals on Windows. A reported 0 ms does not mean no CPU work occurred. Increase `jobs` if measurements are close to the clock's resolution.

## 4. PyTorch has its own CPU thread settings

**Intra-operation threads** divide work inside one operation, such as a matrix
multiplication. **Inter-operation threads** can run independent operations
in execution systems that support it, such as TorchScript, which runs model
operations outside ordinary Python code. This does not mean
ordinary Python statements automatically run together. These CPU thread settings
are separate from DataLoader workers. A PyTorch **tensor** stores numerical data
in one or more dimensions, much like a NumPy array.

The optional helper starts a fresh Python interpreter for each configuration,
sets intra-operation threads before tensor work, and measures a tiny CPU
workload. Fresh interpreters keep each configuration separate. The timed
interval excludes process startup and tensor generation; it is not an
measurement of the whole application from startup to completion.

In the displayed helper, a fixed random **seed** generates the same inputs for
each configuration. `torch.inference_mode()` turns off bookkeeping used for
training **gradients**, numbers that tell training how changing model weights
would change the error. We only want to measure CPU computation. A **checksum** is a
small summary of a result, here the sum of its values. Matching checksums are a
useful quick check but do not prove every value matches.
[PyTorch CPU threading guide](https://docs.pytorch.org/docs/stable/notes/cpu_threading_torchscript_inference.html).
[PyTorch set_num_threads](https://docs.pytorch.org/docs/stable/generated/torch.set_num_threads.html).
[PyTorch inference mode](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html).

In [6]:
import ast
import importlib.util
from IPython.display import Code, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "lessons" / "native_threads_demo.py").is_file())
helper = ROOT / "lessons" / "native_threads_demo.py"
helper_source = helper.read_text(encoding="utf-8")
tree = ast.parse(helper_source)
function = next(node for node in tree.body if getattr(node, "name", None) == "measure_torch")
display(Code(ast.get_source_segment(helper_source, function), language="python"))

def measure_torch(threads):
    import torch

    # Configure CPU operation threads before allocating or computing tensors.
    torch.set_num_threads(threads)
    # A fixed seed generates the same input numbers for every configuration.
    generator = torch.Generator().manual_seed(7)
    left = torch.randn((320, 320), generator=generator, dtype=torch.float64)
    right = torch.randn((320, 320), generator=generator, dtype=torch.float64)
    timings = []
    # This CPU example needs no bookkeeping for training gradients.
    with torch.inference_mode():
        torch.mm(left, right)  # Untimed warm-up in the same mode as the measured work.
        for _ in range(3):
            start = perf_counter()
            for _ in range(8):
                result = torch.mm(left, right)
            timings.append(perf_counter() - start)
    return {
        "threads": torch.get_num_threads(),
        "seconds": statistics.median(timings),
        # A compact comparison of the last result, not a full correctness proof.
        "checksum": result.sum().item(),
    }

In [7]:
if importlib.util.find_spec("torch") is None:
    print("Optional PyTorch section skipped. Install requirements-torch.txt to run it.")
else:
    torch_rows = []
    for requested_threads in (1, 2):
        completed = subprocess.run(
            [sys.executable, str(helper), "--threads", str(requested_threads)],
            cwd=ROOT, capture_output=True, text=True, timeout=90,
        )
        if completed.returncode:
            raise RuntimeError(completed.stdout + "\n" + completed.stderr)
        row = json.loads(completed.stdout)
        torch_rows.append(row)
        print(f"PyTorch threads={row['threads']}: "
              f"median CPU workload elapsed={row['seconds'] * 1000:.2f} ms; "
              f"checksum={row['checksum']:.6f}")
    assert np.isclose(torch_rows[0]["checksum"], torch_rows[1]["checksum"], rtol=1e-5, atol=1e-5)

PyTorch threads=1: median CPU workload elapsed=7.69 ms; checksum=8472.669867


PyTorch threads=2: median CPU workload elapsed=4.04 ms; checksum=8472.669867


**Floating-point** numbers store approximations to real numbers. Results can
differ slightly when operations group arithmetic in a different order. A
**tolerance** allows a small numerical difference in a comparison; `np.isclose`
uses this rule instead of requiring exact equality.
This does not excuse missing samples or a different algorithm.

`torch.set_num_threads()` changes CPU operation threads. It does not set
DataLoader's `num_workers` and does not control CUDA, the NVIDIA GPU computing platform. GPU work has
different scheduling and measurement rules, discussed in lesson 13.

## 5. A practical thread budget

Suppose you start four process workers and each calls a native operation
configured for eight threads. That can request roughly 32 compute threads.
Check the CPUs available to the application, including container or affinity
restrictions. **CPU affinity** restricts which CPUs a process may run on; a
container can also have a CPU time quota. A **logical CPU** is one execution
slot reported by the operating system. A physical core may provide several of
these slots, so their count alone does not describe the available computing power.

A reasonable first experiment is either a few workers with one native thread
each, or one worker using a small native pool. Measure before combining both.
Libraries and operations differ, so a fixed rule cannot replace measurement.

The same native pool limit is applied when each Python worker thread starts;
we avoid changing it independently inside every job. The limit's scope varies
by backend. Some libraries accept environment settings such as `OMP_NUM_THREADS` and
`OPENBLAS_NUM_THREADS`, usually configured before importing the library in
each worker. They control different runtimes. `threadpoolctl` offers a way
to inspect and temporarily control supported pools in a running process.

The native settings in these NumPy experiments are restored when their context
manager exits. The PyTorch settings exist only in each short-lived helper.

## Practice

1. Explain why a NumPy array operation can beat a Python loop without using more cores.
2. Four process workers each request three native threads. What rough compute-thread
   budget does that suggest? What else must you check?
3. Repeat the matrix sweep with size 480 instead of 320. Recreate the arrays and
   `expected_value`, then rerun the measurement cells. Does the winning row change?
4. Why can competing native limits set inside individual jobs be a poor design?
   How does one common worker initializer help?
5. Would `torch.set_num_threads(2)` make a loader use two worker processes?

In [8]:
# Predict before running an additional small configuration.
extra_seconds, extra_cpu_seconds, observed = measure_matrix_configuration(2, 4)
print("Requested 2 Python workers × 4 BLAS threads; detected BLAS limits:", observed)
print(f"Elapsed={extra_seconds * 1000:.2f} ms; CPU={extra_cpu_seconds * 1000:.2f} ms")

Requested 2 Python workers × 4 BLAS threads; detected BLAS limits: [[4]]
Elapsed=16.25 ms; CPU=109.38 ms


<details>
<summary>Worked answers — open after trying</summary>

1. Native code runs the repeated arithmetic with much less Python interpreter overhead.
   Vectorization is not the same as multicore execution.
2. Roughly 12 compute threads, plus coordination. Check actual library behavior,
   allowed CPU capacity, memory bandwidth, and other work in the application.
3. It may change. Larger matrices can make parallel computation more worthwhile,
   but hardware and library implementations still determine the result.
4. The scope varies by backend. Competing changes inside jobs make settings unclear.
   Apply the same setting in a worker initializer and restore settings after the pool ends.
5. No. DataLoader's `num_workers=2` selects two loader processes. CPU operation thread
   count is a separate setting and can multiply resource use inside workers.

</details>

## Explain it back

Distinguish a Python worker, a native computation thread, and a DataLoader
process. Explain why you would inspect all three before increasing parallelism
in an image preprocessing job.

[Next: 12 · Combining concurrency mechanisms](12_combining_concurrency_mechanisms.ipynb).